## Basline Models Checks

**import**

In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

**Load data**

In [16]:
DATASET="../dataset/SmartHomeIoTNLU.csv"

df=pd.read_csv(DATASET)


**Dataset Split**

In [17]:
from sklearn.model_selection import train_test_split

events=df.Event_ID.unique()

train_events,test_events=train_test_split(events,test_size=0.15,random_state=42)

train=df[df.Event_ID.isin(train_events)]

test=df[df.Event_ID.isin(test_events)]


**Intent Classification, TF-IDF + Logistic Regression**

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

vectorizer=TfidfVectorizer()

X_train=vectorizer.fit_transform(train.Command)

X_test=vectorizer.transform(test.Command)

model=LogisticRegression()

model.fit(X_train,train.Intent_Type)

pred=model.predict(X_test)

In [19]:
# Metrics:

from sklearn.metrics import classification_report

print(classification_report(test.Intent_Type,pred))

                       precision    recall  f1-score   support

Direct_Device_Control       0.99      0.99      0.99     19181
        Scene_Control       1.00      1.00      1.00     91866

             accuracy                           1.00    111047
            macro avg       1.00      1.00      1.00    111047
         weighted avg       1.00      1.00      1.00    111047



**Device Multi-label Prediction**

In [20]:
device_events = (
    df.groupby("Event_ID")["Device"]
    .apply(list)
)

device_events.head()

Event_ID
00009a71-b6c4-4723-a5bf-3a3ba165ee86    [Curtains]
0000c720-6640-486c-b8dc-8f7b22127ff6          [AC]
0001368b-b86b-49b9-a7c3-86f4fc860752          [AC]
000166aa-8a13-4ce6-aa0c-57e99c06da80       [Light]
0001899f-d288-4392-82e1-7dbab56e2fbd          [AC]
Name: Device, dtype: object

In [21]:
# Convert devices into multi-label vectors

from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()

Y = mlb.fit_transform(
    device_events
)

print(Y.shape)

(200320, 19)


In [22]:
# Split into train/test

from sklearn.model_selection import train_test_split


X = device_events.index


X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    test_size=0.15,
    random_state=42
)


In [23]:
# Create command features

commands = (
    df.groupby("Event_ID")["Command"]
    .first()
)


from sklearn.feature_extraction.text import TfidfVectorizer


vectorizer = TfidfVectorizer()


X_text = vectorizer.fit_transform(
    commands
)



In [24]:
# Split text features

X_train_text, X_test_text, Y_train, Y_test = train_test_split(
    X_text,
    Y,
    test_size=0.15,
    random_state=42
)

In [25]:
# Train multi-label classifier

from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier


model = OneVsRestClassifier(
    LogisticRegression(
        max_iter=1000
    )
)

model.fit(
    X_train_text,
    Y_train
)

,estimator,LogisticRegre...max_iter=1000)
,n_jobs,None
,verbose,0
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None


In [26]:
# Generate predictions

prediction = model.predict(
    X_test_text
)

In [27]:
#Metrics

from sklearn.metrics import f1_score


f1 = f1_score(
    Y_test,
    prediction,
    average="samples"
)


print(
    "Sample F1:",
    f1
)

Sample F1: 0.9691008922580155


In [30]:
# Add all device prediction metrics

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    hamming_loss
)


results = {

"Precision":
precision_score(
    Y_test,
    prediction,
    average="samples"
),

"Recall":
recall_score(
    Y_test,
    prediction,
    average="samples"
),

"F1-score":
f1_score(
    Y_test,
    prediction,
    average="samples"
),

"Hamming Loss":
hamming_loss(
    Y_test,
    prediction
)

}


results

/home3/ykwx38/myjupyterenv4/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


{'Precision': 0.9751744634307534,
 'Recall': 0.9681913708296496,
 'F1-score': 0.9691008922580155,
 'Hamming Loss': 0.010984179698447397}

## Temperature Parameter Prediction

In [31]:
# Filter AC records

import pandas as pd

# Load dataset
df = pd.read_csv("../dataset/SmartHomeIoTNLU.csv")

# Keep only AC rows
ac_df = df[df["Device"] == "AC"].copy()

print(f"Number of AC samples: {len(ac_df)}")

ac_df.head()

Number of AC samples: 105520


,Event_ID,Intent_Type,Command,Device,Current State,Time,Light,Temperature,Noise,Occupancy,Location,Scenario,Action,Parameters
7,157f097b-0aa5-4aa3-b9b1-b1980b3c7ebd,Scene_Control,We are done cleaning,AC,Off,Noon,88%,23°C,Medium,2,Bedroom,Cleaning_stop_Bedroom,Adjust,Temp=25°C
17,6afbb8f7-b18c-4285-9e26-260e44b337b8,Scene_Control,Turn everything off in here,AC,Off,Morning,75%,19°C,Low,1,Dining Room,Shutdown_Dining Room,Off,Temp=19°C
27,97b8bf74-f736-45b9-aee6-e20987e0092f,Direct_Device_Control,Disable cooling system,AC,Off,Afternoon,89%,20°C,Medium,1,Kitchen,Direct_Control,Off,Temp=20°C
29,e935ecdb-7208-4e11-9c89-445a12f42530,Scene_Control,Kitchen power off,AC,Adjust,Evening,41%,19°C,Low,1,Kitchen,Shutdown_Kitchen,Off,Temp=19°C
54,6552c508-a3fe-4c70-9c4c-c15418504f75,Scene_Control,Launch shutdown mode,AC,Off,Evening,80%,26°C,Low,1,Kitchen,Shutdown_Kitchen,Off,Temp=26°C


In [32]:
# Convert the parameter to a numeric temperature

ac_df["Target_Temperature"] = (
    ac_df["Parameters"]
    .str.extract(r'(\d+)')
    .astype(float)
)

ac_df[["Parameters", "Target_Temperature"]].head()

,Parameters,Target_Temperature
7,Temp=25°C,25.0
17,Temp=19°C,19.0
27,Temp=20°C,20.0
29,Temp=19°C,19.0
54,Temp=26°C,26.0


In [34]:
# Create input features

from sklearn.feature_extraction.text import TfidfVectorizer

# Combine command and context
ac_df["Input_Text"] = (
    ac_df["Command"] + " " +
    ac_df["Time"] + " " +
    ac_df["Location"] + " " +
    ac_df["Noise"]
)

vectorizer = TfidfVectorizer(max_features=1000)

X_text = vectorizer.fit_transform(ac_df["Input_Text"])

In [35]:
# Add numerical features

import numpy as np

# Light: "43%" -> 43
ac_df["Light"] = (
    ac_df["Light"]
    .str.replace("%", "", regex=False)
    .astype(float)
)

# Temperature: "16°C" -> 16
ac_df["Temperature"] = (
    ac_df["Temperature"]
    .str.extract(r'(\d+)')
    .astype(float)
)

In [36]:
# Encode categorical variables

from sklearn.preprocessing import LabelEncoder

time_encoder = LabelEncoder()
room_encoder = LabelEncoder()
noise_encoder = LabelEncoder()

ac_df["Time_ID"] = time_encoder.fit_transform(ac_df["Time"])
ac_df["Room_ID"] = room_encoder.fit_transform(ac_df["Location"])
ac_df["Noise_ID"] = noise_encoder.fit_transform(ac_df["Noise"])

In [37]:
# Combine all features

from scipy.sparse import hstack

numeric = ac_df[
    [
        "Temperature",
        "Light",
        "Occupancy",
        "Time_ID",
        "Room_ID",
        "Noise_ID"
    ]
].values

X = hstack([X_text, numeric])

y = ac_df["Target_Temperature"]

In [38]:
# Train/Test split

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=42
)

In [39]:
# Train a baseline regressor
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

,n_estimators,200
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [40]:
# Predict
prediction = model.predict(X_test)

In [41]:
# Evaluate

from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(y_test, prediction)

rmse = np.sqrt(
    mean_squared_error(y_test, prediction)
)

print(f"MAE  : {mae:.3f}")
print(f"RMSE : {rmse:.3f}")

MAE  : 0.683
RMSE : 1.095


In [42]:
# Display prediction examples

results = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": prediction
})

results.head(20)

,Actual,Predicted
0,23.0,24.39500
1,24.0,21.96000
2,13.0,13.00000
3,18.0,18.07500
4,25.0,24.83000
5,24.0,23.79000
6,25.0,23.02000
7,26.0,25.95500
8,27.0,27.41500
9,24.0,22.41500
